In [ ]:
# ===================== SETTINGS =====================
PROJECT_ROOT = "/Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master"

CKPT_PATH = f"{PROJECT_ROOT}/models/rqvae/best_model.pth"
EMB_PATH  = f"{PROJECT_ROOT}/data/embeds/Pet_Supplies_items_with_embeddings.parquet"
OUT_PATH  = f"{PROJECT_ROOT}/data/embeds/Pet_Supplies_items_with_semantic_ids.parquet"

BATCH_SIZE = 2048
DEVICE = "mps"   # "cuda" / "cpu" / "mps"

# ===================================================

import sys
from pathlib import Path
import numpy as np
import polars as pl
import torch

sys.path.insert(0, PROJECT_ROOT)

from src.rqvae.model import RQVAE, RQVAEConfig
from src.rqvae.train import load_checkpoint
from src.rqvae.evaluate import encode_dataset

In [ ]:
def pick_device(device: str) -> str:
    if device == "cuda" and torch.cuda.is_available():
        torch.set_float32_matmul_precision("high")
        return "cuda"
    if device == "mps" and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def load_rqvae(ckpt_path: str, device: str):
    """Load RQ-VAE from checkpoint using src/rqvae utilities."""
    ckpt = load_checkpoint(Path(ckpt_path))
    cfg = RQVAEConfig(**ckpt.config) if isinstance(ckpt.config, dict) else ckpt.config
    model = RQVAE(cfg)
    # torch.compile adds "_orig_mod." prefix — strip it
    sd = ckpt.model_state_dict
    if any(k.startswith("_orig_mod.") for k in sd):
        sd = {k.replace("_orig_mod.", "", 1): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=True)
    return model.to(device).eval(), cfg


def build_D_disambiguation(df_abc: pl.DataFrame, key_col: str) -> pl.DataFrame:
    """D = ordinal index within (A,B,C) group, sorted by key_col for determinism."""
    df_abc = df_abc.sort(["A", "B", "C", key_col])
    return df_abc.with_columns(
        pl.int_range(0, pl.len()).over(["A", "B", "C"]).cast(pl.Int16).alias("D")
    )


def add_sid_tokens(df_abcd: pl.DataFrame) -> pl.DataFrame:
    return df_abcd.with_columns(
        pl.concat_str([
            pl.lit("<|sid_start|>"),
            pl.concat_str([pl.lit("<|A"), pl.col("A").cast(str), pl.lit("|>")]),
            pl.concat_str([pl.lit("<|B"), pl.col("B").cast(str), pl.lit("|>")]),
            pl.concat_str([pl.lit("<|C"), pl.col("C").cast(str), pl.lit("|>")]),
            pl.concat_str([pl.lit("<|D"), pl.col("D").cast(str), pl.lit("|>")]),
            pl.lit("<|sid_end|>"),
        ], separator="").alias("sid_tokens")
    )

In [ ]:
# --- Encode items → A,B,C,D + sid_tokens ---
device = pick_device(DEVICE)
print("Device:", device)

print("Reading embeddings:", EMB_PATH)
df = pl.read_parquet(EMB_PATH)
emb = torch.tensor(df["embedding"].to_list(), dtype=torch.float32)
print("Embeddings shape:", emb.shape)

key_col = "parent_asin" if "parent_asin" in df.columns else None
if key_col is None:
    df = df.with_row_index("row_id")
    key_col = "row_id"
    print("WARNING: no parent_asin column, using row_id")

print("Loading RQ-VAE:", CKPT_PATH)
model, cfg = load_rqvae(CKPT_PATH, device)
print(f"RQ-VAE: levels={cfg.codebook_quantization_levels}, codebook_size={cfg.codebook_size}")

print("Encoding A/B/C...")
sids, _ = encode_dataset(model, emb, batch_size=BATCH_SIZE)

if sids.shape[1] < 3:
    raise ValueError(f"Model returned only {sids.shape[1]} levels, expected >= 3")

out = pl.DataFrame({key_col: df[key_col]}).with_columns([
    pl.Series("A", sids[:, 0].astype(np.int16)),
    pl.Series("B", sids[:, 1].astype(np.int16)),
    pl.Series("C", sids[:, 2].astype(np.int16)),
])

print("Building D (disambiguation)...")
out = build_D_disambiguation(out, key_col=key_col)
max_d = out.select(pl.col("D").max()).item()
print(f"max D: {max_d} | codebook_size: {cfg.codebook_size}")
if max_d >= cfg.codebook_size:
    print("WARNING: max D >= codebook_size — won't fit in <|D0..D255|> tokens")

out = add_sid_tokens(out)

n = len(out)
n_unique = out.select(pl.struct(["A", "B", "C", "D"]).n_unique()).item()
print(f"Unique (A,B,C,D): {n_unique} / {n} | collisions: {n - n_unique}")

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
out.write_parquet(OUT_PATH)
print("Saved:", OUT_PATH)
out.head(5)

In [7]:
# ===================== SETTINGS =====================
PROJECT_ROOT = "/Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master"

SID_PARQUET  = f"{PROJECT_ROOT}/data/embeds/Pet_Supplies_items_with_semantic_ids.parquet"
SEQS_PARQUET = f"{PROJECT_ROOT}/data/prepared/Pet_Supplies_sequences.parquet"

OUT_ALL   = f"{PROJECT_ROOT}/data/sequences/Pet_Supplies_sequences_with_sid.parquet"
OUT_TRAIN = f"{PROJECT_ROOT}/data/sequences/Pet_Supplies_sequences_with_sid_train.parquet"
OUT_VAL   = f"{PROJECT_ROOT}/data/sequences/Pet_Supplies_sequences_with_sid_val.parquet"

SEQUENCE_COL = "sequence"     # list[str] of ASINs
USER_COL     = "user_id"      # optional, if exists

MIN_LEN = 5    # согласовано с MIN_SEQ_LENGTH в prepape_data.ipynb
MAX_LEN = 100  # согласовано с MAX_SEQUENCE_LENGTH в prepape_data.ipynb

VAL_SPLIT = 0.10
SEED = 42

# what to store as tokens in sequence:
#   "sid_tokens"  -> "<|sid_start|>...<|sid_end|>"
#   "ABCD"        -> ["A","B","C","D"] as struct/list (compact)
SEQ_OUTPUT_FORMAT = "sid_tokens"  # or "ABCD"

# ====================================================

import polars as pl

print("Loading SID table:", SID_PARQUET)
sid = pl.read_parquet(SID_PARQUET)

needed = {"parent_asin", "A", "B", "C", "D", "sid_tokens"}
missing = needed - set(sid.columns)
if missing:
    raise ValueError(f"SID parquet missing columns: {missing}")

sid = sid.select(["parent_asin", "A", "B", "C", "D", "sid_tokens"])
print("SID rows:", sid.height)

print("Loading sequences:", SEQS_PARQUET)
seqs = pl.read_parquet(SEQS_PARQUET)
if SEQUENCE_COL not in seqs.columns:
    raise ValueError(f"Sequences parquet must have column '{SEQUENCE_COL}' (list[str])")

print("Sequences rows:", seqs.height)
print("Sequences columns:", seqs.columns)

# ---- Build mapping dicts (fast python dict) ----
asin2sid = dict(zip(sid["parent_asin"].to_list(), sid["sid_tokens"].to_list()))
asin2abcd = dict(zip(
    sid["parent_asin"].to_list(),
    list(zip(sid["A"].to_list(), sid["B"].to_list(), sid["C"].to_list(), sid["D"].to_list()))
))

# ---- Map function ----
def map_seq_to_sid(seq):
    out = []
    if seq is None:
        return out
    for asin in seq:
        if asin in asin2sid:
            if SEQ_OUTPUT_FORMAT == "sid_tokens":
                out.append(asin2sid[asin])
            else:  # "ABCD"
                a,b,c,d = asin2abcd[asin]
                out.append([int(a), int(b), int(c), int(d)])
    return out

return_dtype = pl.List(pl.Utf8) if SEQ_OUTPUT_FORMAT == "sid_tokens" else pl.List(pl.List(pl.Int16))

print("Mapping sequences to semantic IDs...")
seqs = seqs.with_columns(
    pl.col(SEQUENCE_COL)
      .map_elements(map_seq_to_sid, return_dtype=return_dtype)
      .alias("sid_sequence")
)

seqs = seqs.with_columns(
    pl.col("sid_sequence").list.len().alias("sid_sequence_length")
)

# ---- Filter lengths ----
seqs_non_empty = seqs.filter(pl.col("sid_sequence_length") > 0)
seqs_valid = (
    seqs_non_empty
    .filter(pl.col("sid_sequence_length") >= MIN_LEN)
    .with_columns(
        # cap to MAX_LEN deterministically (take last MAX_LEN; можно поменять на head)
        pl.when(pl.col("sid_sequence_length") > MAX_LEN)
          .then(pl.col("sid_sequence").list.tail(MAX_LEN))
          .otherwise(pl.col("sid_sequence"))
          .alias("sid_sequence")
    )
    .with_columns(pl.col("sid_sequence").list.len().alias("sid_sequence_length"))
)

print("\n=== Mapping stats ===")
print("Original sequences:", seqs.height)
print("Non-empty:", seqs_non_empty.height, f"({seqs_non_empty.height/seqs.height:.1%})")
print(f"Valid (len>={MIN_LEN}):", seqs_valid.height, f"({seqs_valid.height/seqs.height:.1%})")
print("Mean sid length:", float(seqs_valid["sid_sequence_length"].mean()))
print("Max sid length:", int(seqs_valid["sid_sequence_length"].max()))

# ---- Save all mapped ----
Path(OUT_ALL).parent.mkdir(parents=True, exist_ok=True)
print("Saving ALL:", OUT_ALL)
seqs_valid.write_parquet(OUT_ALL)

# ---- Train/val split ----
n_total = seqs_valid.height
n_val = int(n_total * VAL_SPLIT)
n_train = n_total - n_val

seqs_shuf = seqs_valid.sample(fraction=1.0, seed=SEED, shuffle=True)
train_df = seqs_shuf.head(n_train)
val_df   = seqs_shuf.tail(n_val)

print("\n=== Split ===")
print("Total:", n_total)
print("Train:", train_df.height, f"({train_df.height/n_total:.1%})")
print("Val  :", val_df.height, f"({val_df.height/n_total:.1%})")

print("Saving TRAIN:", OUT_TRAIN)
train_df.write_parquet(OUT_TRAIN)

print("Saving VAL:", OUT_VAL)
val_df.write_parquet(OUT_VAL)

print("\nDone.")

Loading SID table: /Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data/embeds/Pet_Supplies_items_with_semantic_ids.parquet
SID rows: 63319
Loading sequences: /Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data/prepared/Pet_Supplies_sequences.parquet
Sequences rows: 101926
Sequences columns: ['user_id', 'sequence', 'seq_len']
Mapping sequences to semantic IDs...

=== Mapping stats ===
Original sequences: 101926
Non-empty: 101926 (100.0%)
Valid (len>=5): 101926 (100.0%)
Mean sid length: 7.550173655397053
Max sid length: 100
Saving ALL: /Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data/sequences/Pet_Supplies_sequences_with_sid.parquet

=== Split ===
Total: 101926
Train: 91734 (90.0%)
Val  : 10192 (10.0%)
Saving TRAIN: /Users/kalistratov/Documents/PYTHON PROJECTS/EDUCATION PROJECTS/MIPT_MASTER/mipt_master/data/sequences/Pet_Supplies_sequences_with_sid_train.parquet
S